# 03 Gold Layer
**CSE 487 Phase 3, Team CTRL-Freaks** (Gold layer by Meena Kasina)

**Purpose:** Turn the clean Silver table into analytics-ready Gold tables for our research question: *are 311 response times fair across Buffalo's council districts and neighborhoods from 2020 to 2026?* Gold rebuilds everything our Phase 1 EDA and Phase 2 models needed, now as Spark tables: one row-level feature table for the models, and summary tables for the EDA, MapReduce, and k-Means work.

**Input:** `workspace.ctrl_freaks.silver_311` from `02_silver_layer`. One table also reads the Bronze tables (`gold_census_tract_summary`), because Silver dropped the census tract column.

**Outputs (Delta tables in `workspace.ctrl_freaks`):**

| Table | One row per | Replaces / feeds | Audience |
|---|---|---|---|
| `gold_311_features` | closed request | Model input for Linear Regression (A), Logistic Regression + Naive Bayes (B), Random Forest (C) | `04_mllib_models` |
| `gold_neighborhood_summary` | neighborhood | k-Means input (C): `totalRequests`, `medianDays` | `04_mllib_models`, report |
| `gold_district_year_summary` | district x year | MapReduce medians by district and year (A) | Report, city-council audience |
| `gold_district_period_summary` | district x before/after May 2024 | MapReduce before/after switch comparison (A) | Report |
| `gold_district_summary` | council district | EDA: requests per district, plus a type-adjusted fairness index | Report |
| `gold_department_summary` | department (`Subject`) | EDA: resolution time by department | Report |
| `gold_department_type_summary` | department x request category | EDA: department vs request type heatmap | Report |
| `gold_type_year_summary` | year x request category | EDA: top 5 requests per year | Report |
| `gold_monthly_trend` | year-month | EDA: monthly request trends | Report |
| `gold_census_tract_summary` | census tract | EDA: top census tracts | Report |
| `gold_address_summary` | address (top 100) | EDA: most frequent addresses | Report |

**Assumptions:** `02_silver_layer` already ran. Everything stays in Spark DataFrames. The only `toPandas()` calls are on the small Gold summary tables, for charts.

**How to run:** Run all, top to bottom. The settings in the first code cell are the only things to change.

## Setup
**What:** Name the catalog and schema, set the Gold settings, and read Silver.

**Why:** The settings that affect the model targets live in one cell so the team can match them to Phase 2.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# same names as bronze and silver, change here if ur catalog is different
catalogName = "workspace"
schemaName = "ctrl_freaks"
silverTable = f"{catalogName}.{schemaName}.silver_311"

# Gold settings (these affect the model targets, so keep them documented)
trainPercent = 80                 # % of requests in the train split (Phase 2 used 80/20 everywhere)
slowQuantile = 0.5                # isSlow = longer than this quantile of its request category (0.5 = median, same as Phase 2)
switchDate = "2024-05-11"         # Buffalo's 311 system switch, same as the Phase 2 afterSwitch feature
dropNegativeDurations = False     # Phase 2 kept closed cases with a negative duration (Phase 1 EDA dropped them). False = match Phase 2
epsilonDays = 0.01                # floor for the category median so relative time never divides by zero (~14 min)

silver = spark.table(silverTable)
silverRows = silver.count()
print("silver rows:", silverRows)
silver.printSchema()

## Step 1: Response Time
**What:** Create `resolutionDays` (days between `OpenDate` and `ClosedDate`) and flag which rows have a usable one.

**Why:** Response time is the core of the project, and Silver left it for Gold. Same as Phase 2: a request has a response time if it has a `ClosedDate`. Open cases have none, so they stay in the counts and summaries but cannot be a model target.

**Negative durations:** if any closed case closed before it opened, the check below shows how many. Phase 2 kept them and the Phase 1 EDA dropped them, so `dropNegativeDurations` in the settings picks which one to follow (default matches Phase 2).

`resolutionDaysValid` is `resolutionDays` for rows with a response time and null otherwise, so Spark's averages and percentiles skip the rest on their own.

In [0]:
df = silver.withColumn("resolutionDays", (F.unix_timestamp("ClosedDate") - F.unix_timestamp("OpenDate")) / 86400)

hasResolution = F.col("ClosedDate").isNotNull()
if dropNegativeDurations:
    hasResolution = hasResolution & (F.col("resolutionDays") >= 0)

df = df.withColumn("hasValidResolution", hasResolution)
df = df.withColumn("resolutionDaysValid", F.when(F.col("hasValidResolution"), F.col("resolutionDays")))

display(df.agg(
    F.count("*").alias("totalRows"),
    F.sum(F.col("isOpen").cast("int")).alias("openCases"),
    F.sum(F.when(F.col("ClosedDate").isNotNull() & (F.col("resolutionDays") < 0), 1).otherwise(0)).alias("negativeDuration"),
    F.sum(F.col("hasValidResolution").cast("int")).alias("closedWithResolution"),
))

# how skewed is it? (a few very long cases drag the mean way above the median, so summaries use medians)
display(df.agg(
    F.round(F.avg("resolutionDaysValid"), 2).alias("meanDays"),
    F.round(F.percentile("resolutionDaysValid", 0.5), 2).alias("medianDays"),
    F.round(F.percentile("resolutionDaysValid", 0.9), 2).alias("p90Days"),
    F.round(F.percentile("resolutionDaysValid", 0.99), 2).alias("p99Days"),
    F.round(F.max("resolutionDaysValid"), 2).alias("maxDays"),
))

## Step 2: Time Features (same as the Phase 2 shared features)
**What:** Add `openYear`, `openMonth`, `season`, and `afterSwitch` (the Phase 2 shared features), plus the extra time features Logistic Regression and Naive Bayes used: `openDayOfWeek`, `openHour`, `isWeekend`.

**Why:** These are all known when a request is opened, so they are safe to predict from. They are written the same way as in Phase 2:
- `season`: Dec-Feb Winter, Mar-May Spring, Jun-Aug Summer, Sep-Nov Fall
- `afterSwitch`: opened on or after May 11, 2024, when Buffalo switched 311 systems (1/0)
- `openDayOfWeek`: **0 = Monday ... 6 = Sunday**, like pandas `dt.dayofweek` in Phase 2. Spark's own `dayofweek` starts at Sunday = 1, so it is shifted here.
- `isWeekend`: Saturday or Sunday (`openDayOfWeek >= 5`)

In [0]:
month = F.month("OpenDate")
season = (F.when(month.isin(12, 1, 2), "Winter")
           .when(month.isin(3, 4, 5), "Spring")
           .when(month.isin(6, 7, 8), "Summer")
           .otherwise("Fall"))

# Spark: Sunday = 1 to Saturday = 7. (x + 5) % 7 turns that into pandas style Monday = 0 to Sunday = 6
dayOfWeek = (F.dayofweek("OpenDate") + 5) % 7

df = (df.withColumn("openYear", F.year("OpenDate"))
        .withColumn("openMonth", month)
        .withColumn("season", season)
        .withColumn("afterSwitch", (F.col("OpenDate") >= F.lit(switchDate).cast("timestamp")).cast("int"))
        .withColumn("openDayOfWeek", dayOfWeek)
        .withColumn("openHour", F.hour("OpenDate"))
        .withColumn("isWeekend", (dayOfWeek >= 5).cast("int")))

display(df.select("CaseReference", "OpenDate", "openYear", "openMonth", "season", "afterSwitch", "openDayOfWeek", "openHour", "isWeekend").limit(5))

## Step 3: Train/Test Split and the `isSlow` Labels
**What:** Give every request a fixed `train` or `test` label, then define the slow label two ways.

**Why:**
- *Split:* hashing `CaseReference` gives the same split on every run and every cluster size (`randomSplit` can change with partitioning). All Spark models read this column, so they all see the same rows. Phase 2 used three different sklearn random splits, which can't be reproduced in Spark, so this is the same *kind* of split (80/20) but not the same rows.
- *`isSlow` (train-only thresholds):* slower than the median of its own request category, with the medians computed on **training rows only** and applied to test rows. This is exactly what Logistic Regression and Naive Bayes did in Phase 2 (B), and it keeps test information out of the label.
- *`isSlowShared` (all-data thresholds):* slower than the median of its category computed on **all** closed rows. This is the Phase 2 shared feature that Random Forest (C) used. It is kept so that model can be compared fairly with its Phase 2 result.

Medians use `F.percentile` (exact, like pandas), not `percentile_approx`, so the thresholds match Phase 2.

In [0]:
df = df.withColumn("split", F.when(F.pmod(F.hash("CaseReference"), F.lit(100)) < trainPercent, "train").otherwise("test"))

hasRes = F.col("hasValidResolution")

# thresholds from training rows only (Phase 2 B)
validTrain = df.filter(hasRes & (F.col("split") == "train"))
typeThresholds = validTrain.groupBy("Type").agg(F.percentile("resolutionDays", slowQuantile).alias("slowThresholdDays"))
globalThreshold = float(validTrain.agg(F.percentile("resolutionDays", slowQuantile)).first()[0])

# thresholds from all closed rows (Phase 2 shared feature, used by Random Forest in C)
typeMedianAll = df.filter(hasRes).groupBy("Type").agg(F.percentile("resolutionDays", 0.5).alias("typeMedianDays"))

df = df.join(typeThresholds, on="Type", how="left").join(typeMedianAll, on="Type", how="left")
# a category with no training rows falls back to the overall training median (same fallback as Phase 2 B)
df = df.withColumn("slowThresholdDays", F.coalesce(F.col("slowThresholdDays"), F.lit(globalThreshold)))

df = df.withColumn("isSlow", F.when(hasRes, (F.col("resolutionDays") > F.col("slowThresholdDays")).cast("int")))
df = df.withColumn("isSlowShared", F.when(hasRes, (F.col("resolutionDays") > F.col("typeMedianDays")).cast("int")))

print("overall training median (days):", globalThreshold)
display(typeThresholds.join(typeMedianAll, on="Type").orderBy(F.desc("slowThresholdDays")))

## Step 4: Type-Adjusted Response Time (for the summaries)
**What:** `relativeTime` = a request's response time divided by the median for its request category. 1.0 means typical for that kind of request, 2.0 means twice as long.

**Why:** This is the Spark version of what Linear Regression showed in Phase 2: after taking out request type, are some districts slower? A district's raw median can look worse just because of the mix of requests it gets (Phase 2: UNIVERSITY had the slowest raw median mostly because of its request types). The median `relativeTime` of a district asks whether it waits longer than the city for the *same kind* of request.

**Not used for modeling:** it uses all-data medians, so it is only for describing the data.

In [0]:
df = df.withColumn("relativeTime", F.when(hasRes, F.col("resolutionDays") / F.greatest(F.col("typeMedianDays"), F.lit(epsilonDays))))

cityMedianDays = df.agg(F.percentile("resolutionDaysValid", 0.5)).first()[0]
print("citywide median response time (days):", cityMedianDays)

## Gold Table: `gold_311_features`
**What:** One row per closed request, with the targets and every input feature the Phase 2 models used.

**Who uses it:** `04_mllib_models`. It is one table for all three supervised models, and the dictionary below says which columns each model takes, copied from the Phase 2 notebooks:

| Phase 2 model | Target | Categorical features (index + one-hot) | Numeric features |
|---|---|---|---|
| Linear Regression (A) | `logResolutionDays` (also `resolutionDays` for the raw version) | `CouncilDistrict`, `Type`, `openMonth` | `afterSwitch` |
| Logistic Regression (B) and Naive Bayes (B) | `isSlow` | `Neighborhood`, `season`, `Type`, `Subject`, `openDayOfWeek`, `openHour`, `openMonth`, `isWeekend`, `Zipcode` | `Latitude`, `Longitude` (median-imputed and scaled, fit on train only) |
| Random Forest (C) | `isSlowShared` | `CouncilDistrict`, `Type`, `Subject`, `season`, `afterSwitch`, `openYear`, `openMonth` | none |
| k-Means (C) | none (unsupervised) | uses `gold_neighborhood_summary`, not this table | |

**Missing values:** missing `Neighborhood` and `Zipcode` become `"Unknown"` (same as Phase 2 B). `Latitude`/`Longitude` stay null because about 55% are missing (old-system placeholder coordinates), and the pipeline in 04 should median-impute them on the training rows.

**`logResolutionDays`:** `log1p` of the response time, since waits are very right-skewed (median about 4 days, max about 1,400). Negative durations are set to 0 first so the log is defined.

**Leakage check:** `ClosedDate`, `Status`, and `isOpen` are only known after a request is resolved, so none of them are features. The check below fails the notebook if any model's feature list contains a post-resolution column, a target, or an ID.

In [0]:
# which columns each Phase 2 model used (4 MLlib models in 04 read from this)
modelFeatureMap = {
    "LinearRegression": {
        "target": "logResolutionDays",
        "categorical": ["CouncilDistrict", "Type", "openMonth"],
        "numeric": ["afterSwitch"]},
    "LogisticRegression_and_NaiveBayes": {
        "target": "isSlow",
        "categorical": ["Neighborhood", "season", "Type", "Subject", "openDayOfWeek", "openHour", "openMonth", "isWeekend", "Zipcode"],
        "numeric": ["Latitude", "Longitude"]},
    "RandomForest": {
        "target": "isSlowShared",
        "categorical": ["CouncilDistrict", "Type", "Subject", "season", "afterSwitch", "openYear", "openMonth"],
        "numeric": []},
}

featureCols = ["CouncilDistrict", "Type", "Subject", "Neighborhood", "Zipcode", "season",
               "Latitude", "Longitude", "openYear", "openMonth", "openDayOfWeek", "openHour", "isWeekend", "afterSwitch"]

goldFeatures = (df.filter(F.col("hasValidResolution"))
                  .fillna("Unknown", subset=["Neighborhood", "Zipcode"])
                  .select("CaseReference", "split",
                          "isSlow", "isSlowShared",
                          "resolutionDays", F.log1p(F.greatest(F.col("resolutionDays"), F.lit(0.0))).alias("logResolutionDays"),
                          *featureCols))

# eakage check: nothing that is only known after the request closes, no targets, no IDs 
notKnownAtPredictionTime = ["ClosedDate", "Status", "isOpen", "resolutionDays", "logResolutionDays", "isSlow", "isSlowShared",
                            "slowThresholdDays", "typeMedianDays", "relativeTime", "CaseReference", "split"]
for modelName, spec in modelFeatureMap.items():
    modelFeatures = spec["categorical"] + spec["numeric"]
    leaked = [c for c in modelFeatures if c in notKnownAtPredictionTime]
    assert len(leaked) == 0, f"{modelName} leakage: {leaked}"
    missingCols = [c for c in modelFeatures + [spec["target"]] if c not in goldFeatures.columns]
    assert len(missingCols) == 0, f"{modelName} columns not in the table: {missingCols}"
print("leakage check passed for all models")

# split sizes and class balance (Phase 2: about 50% slow, 80/20 split)
display(goldFeatures.groupBy("split").agg(
    F.count("*").alias("rows"),
    F.round(F.avg("isSlow"), 3).alias("slowShare"),
    F.round(F.avg("isSlowShared"), 3).alias("slowSharedShare"),
    F.round(F.percentile("resolutionDays", 0.5), 2).alias("medianDays")))

## Summary Tables
**What:** One helper builds the same set of response-time metrics for any grouping, so every summary table is defined the same way.

**Metrics:** `totalRequests` (all rows, open cases included), `openRequests`, `closedRequests`, `openRate`, `meanDays`, `medianDays`, `p90Days`, and `medianRelativeTime` (the type-adjusted index from Step 4).

**Why medians:** a few very long cases pull the mean far above the median, so the median and p90 describe a typical and a bad-case wait better. Medians use the exact `F.percentile`, so they match the Phase 2 pandas and MapReduce numbers.

**Open cases:** recent years still have unclosed cases (Phase 2: 8.4% of 2026 requests), which makes their medians look faster than they are. That's why `openRate` sits next to the medians.

In [0]:
def resolutionSummary(frame, groupCols):
    # same metrics for any grouping
    return (frame.groupBy(*groupCols).agg(
                F.count("*").alias("totalRequests"),
                F.sum(F.col("isOpen").cast("int")).alias("openRequests"),
                F.sum(F.col("hasValidResolution").cast("int")).alias("closedRequests"),
                F.round(F.avg("resolutionDaysValid"), 2).alias("meanDays"),
                F.round(F.percentile("resolutionDaysValid", 0.5), 2).alias("medianDays"),
                F.round(F.percentile("resolutionDaysValid", 0.9), 2).alias("p90Days"),
                F.round(F.percentile("relativeTime", 0.5), 3).alias("medianRelativeTime"))
            .withColumn("openRate", F.round(F.col("openRequests") / F.col("totalRequests"), 4)))

# by council district (+ share of all requests and gap vs the citywide median)
goldDistrict = (resolutionSummary(df, ["CouncilDistrict"])
                .withColumn("requestShare", F.round(F.col("totalRequests") / F.sum("totalRequests").over(Window.partitionBy()), 4))
                .withColumn("medianGapVsCityDays", F.round(F.col("medianDays") - F.lit(float(cityMedianDays)), 2))
                .orderBy("CouncilDistrict"))

# by neighborhood: k-Means input from Phase 2 C (Requests = totalRequests, Median Days = medianDays)
# rows with no neighborhood are left out, same as pandas groupby in Phase 2. They are still in the district table
goldNeighborhood = resolutionSummary(df.filter(F.col("Neighborhood").isNotNull()), ["Neighborhood"]).orderBy(F.desc("totalRequests"))

# MapReduce from Phase 2 A: median wait by district and year
goldDistrictYear = resolutionSummary(df, ["CouncilDistrict", "openYear"]).orderBy("CouncilDistrict", "openYear")

# MapReduce from Phase 2 A: district before vs after the May 2024 system switch
goldDistrictPeriod = (resolutionSummary(df.withColumn("period", F.when(F.col("afterSwitch") == 1, "after May 2024").otherwise("before May 2024")),
                                        ["CouncilDistrict", "period"])
                      .orderBy("CouncilDistrict", "period"))

# Phase 1 EDA tables
goldDepartment = resolutionSummary(df, ["Subject"]).orderBy(F.desc("totalRequests"))
goldDepartmentType = resolutionSummary(df, ["Subject", "Type"]).orderBy("Subject", F.desc("totalRequests"))
goldTypeYear = resolutionSummary(df, ["openYear", "Type"]).orderBy("openYear", F.desc("totalRequests"))
goldMonthly = resolutionSummary(df, ["openYear", "openMonth"]).orderBy("openYear", "openMonth")

display(goldDistrict)

## Phase 1 EDA Tables That Need Extra Work

**Most frequent addresses.** Same as Phase 1: build the full address from `AddressNumber` and `AddressLine1`, skip rows with no street, and keep the top 100 (Phase 1 showed the top 10).

**Census tracts.** Silver dropped the tract columns, so this table reads them from Bronze and joins back to Silver by `CaseReference` (unique in Silver). The tract cleaning is the same as Phase 1: the old system writes decimals (`24`, `16.02`), which are multiplied by 100 and padded to 6 digits (`002400`, `001602`). The new system already uses 6 digits. Anything that is not 6 digits becomes missing and is left out of the table. A check confirms the join did not add or remove Silver rows.

In [0]:
# ---- top addresses ----
# Silver keeps the number as text, strip a trailing .0 just in case (Phase 1 had 554.0 style numbers)
addrNum = F.regexp_replace(F.coalesce(F.col("AddressNumber"), F.lit("")), r"\.0$", "")
withAddress = (df.filter(F.col("AddressLine1").isNotNull())
                 .withColumn("fullAddress", F.trim(F.concat_ws(" ", addrNum, F.col("AddressLine1")))))
goldAddress = resolutionSummary(withAddress, ["fullAddress"]).orderBy(F.desc("totalRequests")).limit(100)

# ---- census tracts (read from bronze, silver dropped them) ----
oldBronze = spark.table(f"{catalogName}.{schemaName}.bronze_311_old")
newBronze = spark.table(f"{catalogName}.{schemaName}.bronze_311_new")

# old system: decimal tract like 24 or 16.02 -> times 100, padded to 6 digits
oldTract = (oldBronze.select("CaseReference", F.expr("try_cast(CensusTract as double)").alias("tractNum"))
                     .withColumn("CensusTract", F.when(F.col("tractNum").isNotNull(),
                                                       F.lpad(F.round(F.col("tractNum") * 100).cast("long").cast("string"), 6, "0")))
                     .select("CaseReference", "CensusTract"))
# new system: already 6 digit text, and its case id column is called CaseNumber
newTract = newBronze.select(F.col("CaseNumber").alias("CaseReference"), "CensusTract")

tracts = (oldTract.unionByName(newTract)
                  .withColumn("CensusTract", F.when(F.col("CensusTract").rlike("^[0-9]{6}$"), F.col("CensusTract")))
                  .dropDuplicates(["CaseReference"]))

withTract = df.join(tracts, on="CaseReference", how="left")
assert withTract.count() == df.count(), "tract join changed the row count"

print("rows with a valid census tract:", withTract.filter(F.col("CensusTract").isNotNull()).count(), "/", df.count())
goldTract = resolutionSummary(withTract.filter(F.col("CensusTract").isNotNull()), ["CensusTract"]).orderBy(F.desc("totalRequests"))
display(goldTract.limit(10))

## Write the Gold Tables
**What:** Save all eleven tables as Delta and add a one-line comment to each so the purpose shows up in the Catalog.

**Why:** `overwrite` means rerunning replaces the table instead of adding rows twice. Each DataFrame is counted right before it is written, so the Delta check below has something to compare to.

In [0]:
goldWrites = {}   # table name -> rows we wrote

def writeGold(frame, tableName, description):
    fullName = f"{catalogName}.{schemaName}.{tableName}"
    rowsBefore = frame.count()
    frame.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(fullName)
    spark.sql(f"COMMENT ON TABLE {fullName} IS '{description}'")
    goldWrites[fullName] = rowsBefore
    print("wrote", rowsBefore, "rows to", fullName)

writeGold(goldFeatures, "gold_311_features", "Model-ready features, targets and fixed train/test split for closed 311 requests")
writeGold(goldNeighborhood, "gold_neighborhood_summary", "Requests and median response time by neighborhood, k-Means input")
writeGold(goldDistrictYear, "gold_district_year_summary", "Response time stats by council district and year, MapReduce replacement")
writeGold(goldDistrictPeriod, "gold_district_period_summary", "Response time stats by council district before and after the May 2024 system switch")
writeGold(goldDistrict, "gold_district_summary", "Requests and response time by council district with type-adjusted fairness index")
writeGold(goldDepartment, "gold_department_summary", "Requests and response time by department")
writeGold(goldDepartmentType, "gold_department_type_summary", "Requests and response time by department and request category")
writeGold(goldTypeYear, "gold_type_year_summary", "Requests and response time by year and request category")
writeGold(goldMonthly, "gold_monthly_trend", "Monthly request volume and response time")
writeGold(goldTract, "gold_census_tract_summary", "Requests and response time by census tract")
writeGold(goldAddress, "gold_address_summary", "Top 100 most frequent request addresses")

## Delta Checks
A write command alone doesn't prove anything, so every Gold table is read back from the catalog. For each one: the row count matches what we wrote and the format is `delta`. The notebook stops if either check fails.

In [0]:
checkRows = []
for fullName, rowsWritten in goldWrites.items():
    back = spark.table(fullName)
    rowsBack = back.count()
    tableFormat = spark.sql(f"DESCRIBE DETAIL {fullName}").first()["format"]
    checkRows.append((fullName, rowsWritten, rowsBack, "PASS" if rowsBack == rowsWritten and tableFormat == "delta" else "FAIL", len(back.columns), tableFormat))

display(spark.createDataFrame(checkRows, ["table", "rowsWritten", "rowsReadBack", "result", "columns", "format"]))
assert all(r[3] == "PASS" for r in checkRows), "a Gold table did not read back correctly"
print("all Gold tables read back as Delta with matching row counts")

In [0]:
# schema of the model table, this is what 04_mllib_models gets
spark.table(f"{catalogName}.{schemaName}.gold_311_features").printSchema()

In [0]:
# whole pipeline in one table: bronze -> silver -> gold, all read back from the catalog
pipelineRows = []
for tableName in ["bronze_311_old", "bronze_311_new", "silver_311"] + [n.split(".")[-1] for n in goldWrites]:
    fullName = f"{catalogName}.{schemaName}.{tableName}"
    pipelineRows.append((fullName, spark.table(fullName).count(), spark.sql(f"DESCRIBE DETAIL {fullName}").first()["format"]))
display(spark.createDataFrame(pipelineRows, ["table", "rows", "format"]))

# Phase 1 EDA, Rebuilt from the Gold Tables
**What:** Every Phase 1 EDA chart and table, now built from the Gold summaries.

**Why:** Phase 1 charted the full pandas DataFrame. Here the aggregation already happened in Spark, so `toPandas()` only moves a few dozen summary rows into matplotlib, never the full dataset.

## Monthly Service Trends by Year

In [0]:
%pip install seaborn

In [0]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

monthlyPd = spark.table(f"{catalogName}.{schemaName}.gold_monthly_trend").toPandas()
monthNames = ["JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC"]

# all years on one chart
trend = monthlyPd.pivot(index="openMonth", columns="openYear", values="totalRequests").sort_index()
trend.index = [monthNames[m - 1] for m in trend.index]
trend.plot(kind="line", figsize=(12, 6))
plt.title("Monthly Buffalo 311 Service Requests Trends from 2020-2026")
plt.xlabel("Month")
plt.ylabel("Number of cases")
plt.show()

In [0]:
# one chart per year (phase 1 made these one at a time). 2026 is a partial year
fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes = axes.flatten()
for ax, year in zip(axes, range(2020, 2027)):
    d = monthlyPd[monthlyPd["openYear"] == year].sort_values("openMonth")
    ax.plot([monthNames[m - 1] for m in d["openMonth"]], d["totalRequests"])
    ax.set_title(f"Monthly 311 Requests in {year}")
    ax.set_xlabel("Month")
    ax.set_ylabel("Number of cases")
    ax.tick_params(axis="x", rotation=45)
axes[-1].axis("off")
plt.tight_layout()
plt.show()

## Top 5 Service Requests Per Year

In [0]:
typeYearPd = spark.table(f"{catalogName}.{schemaName}.gold_type_year_summary").toPandas()
barColors = ["olivedrab", "orange", "cornflowerblue", "mediumpurple", "hotpink"]

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()
for ax, year in zip(axes, range(2020, 2027)):
    top5 = typeYearPd[typeYearPd["openYear"] == year].nlargest(5, "totalRequests")
    ax.barh(top5["Type"], top5["totalRequests"], color=barColors)
    ax.invert_yaxis()
    ax.set_title(f"Top 5 Service Requests for {year}")
axes[-1].axis("off")
plt.tight_layout()
plt.show()

## Total Requests per District (2020-2026)

In [0]:
districtPd = spark.table(f"{catalogName}.{schemaName}.gold_district_summary").toPandas().sort_values("totalRequests")

plt.figure(figsize=(8, 5))
plt.barh(districtPd["CouncilDistrict"], districtPd["totalRequests"],
         color=["darkseagreen", "lightseagreen", "cornflowerblue", "orange", "mediumpurple", "skyblue", "pink", "salmon", "goldenrod"][:len(districtPd)])
plt.title("Total Requests per District from 2020-2026")
plt.show()

## Resolution Times by Department
How many days each department takes to resolve a request, longest median first (same table as Phase 1).

In [0]:
display(spark.table(f"{catalogName}.{schemaName}.gold_department_summary")
        .select(F.col("Subject"),
                F.col("meanDays").alias("Mean Days to Close"),
                F.col("medianDays").alias("Median Days to Close"),
                F.col("closedRequests").alias("Total Cases"))
        .orderBy(F.desc("Median Days to Close")))

## Top 10 Census Tracts with 311 Service Requests

In [0]:
display(spark.table(f"{catalogName}.{schemaName}.gold_census_tract_summary")
        .select(F.col("CensusTract").cast("int").alias("Census Tract"), F.col("totalRequests").alias("total_requests"))
        .orderBy(F.desc("total_requests"))
        .limit(10))

## Top 10 Addresses That Request 311 Services

In [0]:
display(spark.table(f"{catalogName}.{schemaName}.gold_address_summary")
        .select(F.col("fullAddress").alias("address"), F.col("totalRequests").alias("total_requests"))
        .orderBy(F.desc("total_requests"))
        .limit(10))

## Top 10 Request Types for the Top 5 Departments

In [0]:
deptTypePd = spark.table(f"{catalogName}.{schemaName}.gold_department_type_summary").toPandas()
deptPd = spark.table(f"{catalogName}.{schemaName}.gold_department_summary").toPandas()

top5Depts = deptPd.nlargest(5, "totalRequests")["Subject"]
top10Types = typeYearPd.groupby("Type")["totalRequests"].sum().nlargest(10).index

crosstab = (deptTypePd[deptTypePd["Subject"].isin(top5Depts) & deptTypePd["Type"].isin(top10Types)]
            .pivot(index="Subject", columns="Type", values="totalRequests").fillna(0).astype(int))

plt.figure(figsize=(12, 6))
sns.heatmap(crosstab, annot=True, fmt="d", cmap="Blues", cbar=True)
plt.title("Concentration of Request Types by Department")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# Phase 2 Results, Rebuilt from the Gold Tables
**What:** The two Phase 2 analyses that were aggregations rather than trained models: the MapReduce medians (A) and the k-Means input table (C). These check that the Spark tables line up with what we reported.

**Expected:** the numbers will be close to, but not exactly, the Phase 2 write-ups (for example NORTH slowest in 2026, gap before vs after the switch of 0.71 vs 1.92 days), because the Phase 2 notebooks downloaded the data on earlier dates and the new dataset grows every day.

In [0]:
# MapReduce (A): median wait for every district and year, and the gap between the slowest and fastest district
dyPd = spark.table(f"{catalogName}.{schemaName}.gold_district_year_summary").toPandas()
medTable = dyPd.pivot(index="CouncilDistrict", columns="openYear", values="medianDays").round(2)
display(medTable.reset_index())

In [0]:
medTable = dyPd.pivot(index="CouncilDistrict", columns="openYear", values="medianDays").round(2)
gap = pd.DataFrame({"slowest": medTable.idxmax(), "slowestDays": medTable.max(),
                    "fastest": medTable.idxmin(), "fastestDays": medTable.min()})
gap["gapDays"] = (gap["slowestDays"] - gap["fastestDays"]).round(2)
print(gap)

# % of requests still open by year, recent years look faster than they are
print("\n% of requests still open, by year:")
print((dyPd.groupby("openYear")["openRequests"].sum() / dyPd.groupby("openYear")["totalRequests"].sum() * 100).round(1).to_string())

plt.figure(figsize=(10, 6))
for district in medTable.index:
    plt.plot(medTable.columns, medTable.loc[district], marker="o", label=district)
plt.axvline(2024.4, color="gray", linestyle="--", label="new 311 system (May 2024)")
plt.xlabel("year")
plt.ylabel("median days to resolve")
plt.title("Median wait by council district")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.show()

In [0]:
# MapReduce (A): before vs after the May 2024 system switch
periodPd = spark.table(f"{catalogName}.{schemaName}.gold_district_period_summary").toPandas()
periodTable = periodPd.pivot(index="CouncilDistrict", columns="period", values="medianDays")[["before May 2024", "after May 2024"]].round(2)
periodTable["change"] = (periodTable["after May 2024"] - periodTable["before May 2024"]).round(2)
print("gap before:", round(periodTable["before May 2024"].max() - periodTable["before May 2024"].min(), 2),
      " gap after:", round(periodTable["after May 2024"].max() - periodTable["after May 2024"].min(), 2))
display(periodTable.sort_values("change").reset_index())

In [0]:
# k-Means (C): this is the table the model clusters. Requests = all requests, Median Days = closed requests only
hoodInput = (spark.table(f"{catalogName}.{schemaName}.gold_neighborhood_summary")
             .select("Neighborhood", F.col("totalRequests").alias("Requests"), F.col("medianDays").alias("Median Days")))
print("neighborhoods:", hoodInput.count(), "(Phase 2 had 35)")
display(hoodInput.orderBy("Median Days"))

## Summary

**Tables written (all Delta, `workspace.ctrl_freaks`):** `gold_311_features`, `gold_neighborhood_summary`, `gold_district_year_summary`, `gold_district_period_summary`, `gold_district_summary`, `gold_department_summary`, `gold_department_type_summary`, `gold_type_year_summary`, `gold_monthly_trend`, `gold_census_tract_summary`, `gold_address_summary`. Row counts are in the Delta check table above (copy them into the report).

**What each Phase 2 piece reads from Gold**

| Phase 2 piece | Spark version reads |
|---|---|
| Linear Regression (A) | `gold_311_features`, target `logResolutionDays` |
| Logistic Regression, Naive Bayes (B) | `gold_311_features`, target `isSlow` |
| Random Forest (C) | `gold_311_features`, target `isSlowShared` |
| k-Means (C) | `gold_neighborhood_summary` |
| MapReduce (A) | `gold_district_year_summary`, `gold_district_period_summary` (a Spark `groupBy` with an exact median replaces the hand-written map / shuffle / reduce) |

**Choices to put in the report:**
- Response time = `ClosedDate - OpenDate` in days. Open cases have none, so they are left out of the feature table but still counted in the summaries (`openRequests`, `openRate`).
- Two slow labels on purpose: `isSlow` (category medians from training rows only, as in Phase 2 B) and `isSlowShared` (category medians from all closed rows, the Phase 2 shared feature used by Random Forest).
- Train/test split is a fixed 80/20 hash of `CaseReference`. Phase 2 used three different sklearn random splits (seeds 42, 42, and 0 with stratify), which Spark can't reproduce row for row, so the report should say the splits are the same kind but not the same rows.
- `afterSwitch`, `season`, `openDayOfWeek`, and `isWeekend` are built exactly as in Phase 2 (Monday = 0).
- Leakage: `ClosedDate`, `Status`, `isOpen`, and anything built from the response time are never features. The check covers every model's feature list.

**Limitations:**
- About 55% of rows have no coordinates (old-system placeholder), so `Latitude`/`Longitude` need median imputation in the model pipeline.
- Open cases have no response time, so medians describe closed cases only, and recent years look faster than they are.
- The data snapshot is newer than the one in Phase 2, so counts and metrics won't match the Phase 2 numbers exactly.
- `gold_census_tract_summary` depends on the Bronze tables, since Silver dropped the tract columns.

**For `04_mllib_models`:** read `workspace.ctrl_freaks.gold_311_features`. Fit on `split == "train"`, evaluate on `split == "test"`. Use the `modelFeatureMap` dictionary in this notebook for each model's target and columns: `StringIndexer` + `OneHotEncoder` for categorical, `Imputer` (train only) + `StandardScaler` for `Latitude`/`Longitude`, then `VectorAssembler`.